In [ ]:
from pathlib import Path
import os
import shutil
import glob
import sys
import math
import random

In [ ]:
raw_input_directory = Path("raws/input")
raw_output_directory = Path("raws/output")

raw_input_directory.mkdir(parents=True, exist_ok=True)
raw_output_directory.mkdir(parents=True, exist_ok=True)

for item in raw_output_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

Path(raw_output_directory / "images" / "train").mkdir(parents=True, exist_ok=True)
Path(raw_output_directory / "images" / "test").mkdir(parents=True, exist_ok=True)
Path(raw_output_directory / "images" / "val").mkdir(parents=True, exist_ok=True)

Path(raw_output_directory / "labels" / "train").mkdir(parents=True, exist_ok=True)
Path(raw_output_directory / "labels" / "test").mkdir(parents=True, exist_ok=True)
Path(raw_output_directory / "labels" / "val").mkdir(parents=True, exist_ok=True)

In [ ]:
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-9, "Invalid ratios"

total_images = 0
total_labels = 0

for img in raw_input_directory.glob("*.jpg"):
    total_images += 1

for lbl in raw_input_directory.glob("*.txt"):
    total_labels += 1

print(f"Found {total_images} images.\nFound {total_labels} labels")
if total_images != total_labels:
    raise ValueError(f"Dataset mismatch: images={total_images}, labels={total_labels}")

raw = [
    total_images * train_ratio,
    total_images * val_ratio,
    total_images * test_ratio
]

floored = [math.floor(x) for x in raw]
remainder = total_images - sum(floored)
fractions = [x - math.floor(x) for x in raw]
order = sorted(range(3), key=lambda i: fractions[i], reverse=True)
for i in range(remainder):
    floored[order[i]] += 1
train_amount, val_amount, test_amount = floored

print(f"Train Amount: {train_amount}\nVal Amount: {val_amount}\nTest Amount: {test_amount}")


In [ ]:
image_list = list(raw_input_directory.glob("*.jpg"))
label_list = list(raw_input_directory.glob("*.txt"))

dataset = list(zip(image_list, label_list))
print(f"Dataset Amount: {len(dataset)}")

random.shuffle(dataset)
train_set = dataset[:train_amount]
val_set = dataset[train_amount:train_amount + val_amount]
test_set = dataset[train_amount + val_amount:]

assert len(dataset) == len(train_set) + len(val_set) + len(test_set)

In [ ]:
for data in train_set:
    img, label = data
    shutil.copy(img, raw_output_directory / "images" / "train")
    shutil.copy(img, raw_output_directory / "labels" / "train")

for data in val_set:
    img, label = data
    shutil.copy(img, raw_output_directory / "images" / "val")
    shutil.copy(img, raw_output_directory / "labels" / "val")

for data in test_set:
    img, label = data
    shutil.copy(img, raw_output_directory / "images" / "test")
    shutil.copy(img, raw_output_directory / "labels" / "test")